# Detector de Clima — SAM 3

En este notebook construimos un prototipo local con GPU NVIDIA basado **exclusivamente en SAM 3** (`facebook/sam3`) para **determinar el clima actual a partir de una fotografía** usando el dataset [Weather Image Recognition (Kaggle)](https://www.kaggle.com/datasets/jehanbhathena/weather-dataset) (11 clases, 6,862 imágenes).

### Arquitectura unificada 100% SAM 3
Aprovechamos un solo modelo fundacional (`Sam3Model`) en dos etapas:
1. **Línea base — SAM 3 Zero-Shot (sin entrenamiento):** consulta 11 *prompts* de texto climático reutilizando `vision_embeds` para medir qué tan bien reconoce y segmenta el clima sin haber visto el dataset.
2. **Clasificador sobre el Perception Encoder de SAM 3 (Linear Probe / MLP):** extraemos las representaciones visuales ricas (`last_hidden_state`) del codificador interno de SAM 3 (congelado) mediante **Mean + Max Pooling L2** y entrenamos una cabeza clasificadora ligera en segundos sobre las 11 clases.
3. **Inferencia unificada y explicable (`diagnosticar_clima`):** para cualquier foto nueva, calculamos `vision_embeds` **una sola vez** → la cabeza clasificadora predice el clima con alta precisión → reutilizamos esos mismos `vision_embeds` en el decodificador de SAM 3 para **segmentar visualmente la evidencia climática**.

---

## 0. Verificación de entorno, GPU y acceso a Hugging Face

Verificamos:
- Que PyTorch detecte tu GPU NVIDIA (por ejemplo, RTX 3060 Ti).
- Que `transformers` esté en versión `5.x` (necesaria para `Sam3Model` y `Sam3Processor`).
- Que tu sesión de Hugging Face esté autenticada con acceso aprobado a `facebook/sam3`.

In [ ]:
import os
import re
import time
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
import transformers
from huggingface_hub import HfApi
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

# Reproducibilidad
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Directorios del proyecto (funciona tanto desde la raíz como desde notebooks/)
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
MODELS_DIR = PROJECT_ROOT / "models"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
MODELS_DIR.mkdir(parents=True, exist_ok=True)
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

# Configuración de dispositivo y precisión
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE == "cuda":
    DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(f"GPU detectada: {gpu_name} ({vram_gb:.1f} GB VRAM)")
else:
    DTYPE = torch.float32
    print("AVISO: No se detectó GPU CUDA. El proceso será lento en CPU.")

print(f"PyTorch: {torch.__version__} | Transformers: {transformers.__version__} | Dtype: {DTYPE}")
print(f"Raíz del proyecto: {PROJECT_ROOT}")

# Verificar login en Hugging Face (necesario para facebook/sam3)
try:
    hf_user = HfApi().whoami()
    print(f"Hugging Face autenticado como: {hf_user.get('name', 'usuario')}")
except Exception:
    print(
        "AVISO: No se detectó sesión activa de Hugging Face.\n"
        "1) Acepta la licencia en https://huggingface.co/facebook/sam3\n"
        "2) Ejecuta en tu terminal: hf auth login (o huggingface-cli login)"
    )

## 1. Descarga y exploración del dataset (`kagglehub`)

Descargamos `jehanbhathena/weather-dataset` en caché local, indexamos las 11 carpetas de clima y generamos una partición estratificada reproducible:
- **70% entrenamiento (`train_df`)**
- **15% validación (`val_df`)**
- **15% prueba (`test_df`)**

In [ ]:
import kagglehub

IMG_EXTS = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}

raw_dataset_path = Path(kagglehub.dataset_download("jehanbhathena/weather-dataset"))
print("Dataset descargado en:", raw_dataset_path)

def find_class_root(base_dir: Path) -> Path:
    """Localiza la carpeta que contiene directamente las subcarpetas de clases."""
    def _search(directory: Path) -> Path | None:
        subdirs = [d for d in directory.iterdir() if d.is_dir()]
        dirs_with_images = [
            d for d in subdirs
            if any(f.is_file() and f.suffix.lower() in IMG_EXTS for f in d.iterdir())
        ]
        if len(dirs_with_images) >= 2:
            return directory
        for d in subdirs:
            found = _search(d)
            if found is not None:
                return found
        return None

    return _search(base_dir) or base_dir

DATASET_DIR = find_class_root(raw_dataset_path)
print("Carpeta raíz de clases:", DATASET_DIR)

# Indexar todas las imágenes
records = []
class_dirs = sorted([d for d in DATASET_DIR.iterdir() if d.is_dir()], key=lambda p: p.name.lower())
CLASSES = [d.name for d in class_dirs]
CLASS_TO_IDX = {name: i for i, name in enumerate(CLASSES)}
IDX_TO_CLASS = {i: name for i, name in enumerate(CLASSES)}

for cls_dir in class_dirs:
    for img_path in cls_dir.iterdir():
        if img_path.is_file() and img_path.suffix.lower() in IMG_EXTS:
            records.append({
                "filepath": str(img_path),
                "label": cls_dir.name,
                "label_id": CLASS_TO_IDX[cls_dir.name],
            })

df = pd.DataFrame(records)
print(f"Total de imágenes: {len(df):,} | Total de clases: {len(CLASSES)}")
print("Clases detectadas:", CLASSES)

# Gráfico de distribución por clase
counts = df["label"].value_counts().sort_values(ascending=True)
plt.figure(figsize=(9, 4.8))
counts.plot(kind="barh", color="#3b82f6", edgecolor="black", alpha=0.85)
plt.title("Distribución de imágenes por clase climática")
plt.xlabel("Número de imágenes")
plt.ylabel("Clase")
plt.grid(axis="x", linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()

In [ ]:
# Mostrar 1 ejemplo visual por clase
n_cols = 4
n_rows = int(np.ceil(len(CLASSES) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(14, 3.2 * n_rows))
axes = axes.flatten()

for ax, cls_name in zip(axes, CLASSES):
    sample_path = df[df["label"] == cls_name]["filepath"].sample(1, random_state=SEED).iloc[0]
    img = Image.open(sample_path).convert("RGB")
    ax.imshow(img)
    ax.set_title(cls_name, fontsize=11, fontweight="bold")
    ax.axis("off")

for ax in axes[len(CLASSES):]:
    ax.axis("off")

plt.suptitle("Ejemplos de las 11 clases del Weather Dataset", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

# Partición estratificada 70% train / 15% val / 15% test
train_df, temp_df = train_test_split(
    df, test_size=0.30, stratify=df["label_id"], random_state=SEED
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, stratify=temp_df["label_id"], random_state=SEED
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print(f"Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}")

# Muestra estratificada para evaluar la línea base SAM 3 Zero-Shot rápidamente
SAM_ZEROSHOT_PER_CLASS = 10  # 110 imágenes de test para comparar Zero-Shot vs SAM 3 Classifier

def stratified_cap(frame: pd.DataFrame, per_class: int | None, seed: int = SEED) -> pd.DataFrame:
    if per_class is None:
        return frame.copy().reset_index(drop=True)
    parts = [
        g.sample(min(len(g), per_class), random_state=seed)
        for _, g in frame.groupby("label", sort=True)
    ]
    return pd.concat(parts, ignore_index=True)

sam_zs_test_df = stratified_cap(test_df, SAM_ZEROSHOT_PER_CLASS)
print(f"Muestra para evaluación SAM 3 Zero-Shot: {len(sam_zs_test_df)} imágenes")

## 2. Carga de SAM 3 y configuración de prompts climáticos

Cargamos `facebook/sam3` en media precisión (`bfloat16`/`float16`, ~2 GB VRAM). Este único modelo nos servirá tanto para:
- **Extraer embeddings visuales de alta calidad** (desde su *Perception Encoder* interno: `vision_encoder.backbone`).
- **Segmentar y explicar visualmente** el fenómeno climático mediante sus *prompts* de texto y su *Mask Decoder*.

In [ ]:
from transformers import Sam3Model, Sam3Processor

SAM3_MODEL_ID = "facebook/sam3"

print(f"Cargando {SAM3_MODEL_ID} en {DEVICE} ({DTYPE})...")
sam_processor = Sam3Processor.from_pretrained(SAM3_MODEL_ID)
sam_model = Sam3Model.from_pretrained(SAM3_MODEL_ID, dtype=DTYPE).to(DEVICE)
sam_model.eval()
print("SAM 3 cargado correctamente.")

# Prompts cortos y concretos para que el decodificador de SAM 3 segmente bien cada fenómeno
PROMPT_TEMPLATES = {
    "dew": "water droplets",
    "fogsmog": "fog",
    "frost": "frost",
    "glaze": "ice coating on branches",
    "hail": "hailstones",
    "lightning": "lightning bolt",
    "rain": "rain",
    "rainbow": "rainbow",
    "rime": "frost on tree branches",
    "sandstorm": "dust storm",
    "snow": "snow",
}

SPANISH_LABELS = {
    "dew": "Rocío",
    "fogsmog": "Niebla / Smog",
    "frost": "Escarcha",
    "glaze": "Hielo vítreo (Glaze)",
    "hail": "Granizo",
    "lightning": "Rayos / Tormenta eléctrica",
    "rain": "Lluvia",
    "rainbow": "Arcoíris",
    "rime": "Cencellada (Rime)",
    "sandstorm": "Tormenta de arena",
    "snow": "Nieve",
}

def normalize_key(name: str) -> str:
    return re.sub(r"[^a-z0-9]", "", name.lower())

WEATHER_PROMPTS = {
    cls: PROMPT_TEMPLATES.get(normalize_key(cls), cls.replace("_", " "))
    for cls in CLASSES
}
WEATHER_NAMES_ES = {
    cls: SPANISH_LABELS.get(normalize_key(cls), cls)
    for cls in CLASSES
}

pd.DataFrame([
    {"Clase": c, "Nombre ES": WEATHER_NAMES_ES[c], "Prompt SAM 3": WEATHER_PROMPTS[c]}
    for c in CLASSES
])

## 3. Bloque 1 — Línea base: SAM 3 Zero-Shot (Sin entrenamiento)

Primero medimos hasta dónde llega **SAM 3 sin ningún entrenamiento** usando únicamente *Concept Prompting*:
1. `sam_model.get_vision_features(pixel_values)` codifica la foto **una sola vez**.
2. Se consultan los 11 *prompts* climáticos reutilizando `vision_embeds`.
3. Combinamos `presence_logits` (65%) y `max_mask_score` (35%) para ver qué fenómenos reconoce directamente (*zero-shot*) y en cuáles necesita aprender de los datos.

In [ ]:
def segment_prompt_from_embeds(
    vision_embeds,
    original_sizes: list,
    prompt_text: str,
    score_threshold: float = 0.25,
    mask_threshold: float = 0.5,
) -> dict:
    """Ejecuta el decodificador de SAM 3 para un solo prompt reutilizando vision_embeds."""
    h, w = original_sizes[0]
    text_inputs = sam_processor(text=prompt_text, return_tensors="pt").to(DEVICE)

    with torch.inference_mode(), torch.autocast(device_type="cuda", dtype=DTYPE, enabled=(DEVICE == "cuda")):
        outputs = sam_model(vision_embeds=vision_embeds, **text_inputs)

    presence_logits = getattr(outputs, "presence_logits", None)
    presence_prob = float(torch.sigmoid(presence_logits.float()).mean().item()) if presence_logits is not None else 0.0

    processed = sam_processor.post_process_instance_segmentation(
        outputs,
        threshold=score_threshold,
        mask_threshold=mask_threshold,
        target_sizes=original_sizes,
    )[0]

    masks = processed["masks"]
    scores = processed["scores"]

    if len(scores) > 0:
        max_mask_score = float(scores.float().max().item())
        if isinstance(masks, torch.Tensor):
            m_bool = masks > 0
            if m_bool.ndim == 4:
                m_bool = m_bool.squeeze(1)
            union_mask = m_bool.any(dim=0).detach().cpu().numpy().astype(bool)
        else:
            union_mask = np.any([np.asarray(m) > 0 for m in masks], axis=0).astype(bool)
        area_ratio = float(union_mask.mean())
    else:
        max_mask_score = 0.0
        union_mask = np.zeros((h, w), dtype=bool)
        area_ratio = 0.0

    combined_score = 0.65 * presence_prob + 0.35 * max_mask_score
    return {
        "presence_prob": presence_prob,
        "max_mask_score": max_mask_score,
        "area_ratio": area_ratio,
        "num_instances": int(len(scores)),
        "score": combined_score,
        "mask": union_mask,
    }


def analyze_zeroshot_sam3(image: Image.Image, score_threshold: float = 0.25, return_masks: bool = True) -> dict:
    """Evalúa una imagen contra los 11 prompts climáticos usando SAM 3 Zero-Shot."""
    image_rgb = image.convert("RGB")
    img_inputs = sam_processor(images=image_rgb, return_tensors="pt")
    pixel_values = img_inputs["pixel_values"].to(device=DEVICE, dtype=DTYPE)
    original_sizes = img_inputs["original_sizes"].tolist()

    with torch.inference_mode(), torch.autocast(device_type="cuda", dtype=DTYPE, enabled=(DEVICE == "cuda")):
        vision_embeds = sam_model.get_vision_features(pixel_values=pixel_values)

    per_class = {}
    masks_by_class = {}
    for cls_name, prompt_text in WEATHER_PROMPTS.items():
        res = segment_prompt_from_embeds(vision_embeds, original_sizes, prompt_text, score_threshold=score_threshold)
        if return_masks:
            masks_by_class[cls_name] = res.pop("mask")
        else:
            res.pop("mask")
        per_class[cls_name] = res

    score_vector = np.array([per_class[c]["score"] for c in CLASSES], dtype=np.float32)
    pred_idx = int(np.argmax(score_vector))
    return {
        "predicted_class": CLASSES[pred_idx],
        "predicted_idx": pred_idx,
        "score_vector": score_vector,
        "per_class": per_class,
        "masks": masks_by_class,
    }


def overlay_mask(image: Image.Image, mask: np.ndarray, color=(0, 210, 255), alpha: float = 0.45) -> np.ndarray:
    """Superpone una máscara binaria semitransparente sobre una imagen RGB."""
    img_np = np.array(image.convert("RGB"), dtype=np.float32)
    out = img_np.copy()
    if mask is not None and mask.any():
        color_arr = np.array(color, dtype=np.float32)
        out[mask] = (1.0 - alpha) * out[mask] + alpha * color_arr
    return np.clip(out, 0, 255).astype(np.uint8)


# Evaluación Zero-Shot en sam_zs_test_df (110 imágenes)
zs_cache_path = OUTPUTS_DIR / "sam3_zeroshot_test_scores.npy"
if zs_cache_path.exists():
    sam_zs_scores = np.load(zs_cache_path)
    if len(sam_zs_scores) != len(sam_zs_test_df):
        sam_zs_scores = None
else:
    sam_zs_scores = None

if sam_zs_scores is None:
    t0 = time.time()
    scores_list = []
    for fp in tqdm(sam_zs_test_df["filepath"].tolist(), desc="SAM 3 Zero-Shot (Test)"):
        res = analyze_zeroshot_sam3(Image.open(fp), return_masks=False)
        scores_list.append(res["score_vector"])
    sam_zs_scores = np.vstack(scores_list)
    np.save(zs_cache_path, sam_zs_scores)
    print(f"Zero-Shot completado en {time.time() - t0:.1f} s (guardado en caché)")
else:
    print(f"Scores Zero-Shot cargados desde caché: {zs_cache_path}")

y_zs = sam_zs_test_df["label_id"].values
pred_zs = sam_zs_scores.argmax(axis=1)
zs_acc = accuracy_score(y_zs, pred_zs)
zs_f1 = f1_score(y_zs, pred_zs, average="macro")
print(f"\n=== Línea base: SAM 3 Zero-Shot (sin entrenamiento) ===")
print(f"Accuracy: {zs_acc:.2%} | Macro F1: {zs_f1:.2%}\n")
print(classification_report(y_zs, pred_zs, target_names=CLASSES, zero_division=0))

## 4. Bloque 2 — Extracción de Embeddings del Perception Encoder de SAM 3

En lugar de entrenar una CNN externa, aprovechamos directamente el **Perception Encoder** que vive dentro de **SAM 3** (`sam_model.vision_encoder.backbone`).

### ¿Cómo construimos el vector visual de cada imagen?
De `last_hidden_state` (forma `[B, num_patches, hidden_dim]`) extraemos dos resúmenes complementarios:
1. **Mean Pooling (normalizado L2):** captura el **ambiente global** de la escena (cielo nublado, niebla, nieve cubriendo el suelo, tormenta de arena).
2. **Max Pooling (normalizado L2):** captura **señales locales intensas** que ocupan poca área (un rayo en el cielo, un arcoíris, gotas de rocío o brillo de hielo vítreo).

Al concatenarlos (`[Mean, Max]`) obtenemos un vector compacto de `2 × hidden_dim` dimensiones que se guarda en caché (`outputs/sam3_embeddings.npz`) para entrenar y experimentar en segundos.

In [ ]:
def pool_sam3_hidden_state(last_hidden_state: torch.Tensor) -> torch.Tensor:
    """
    Convierte last_hidden_state [B, seq_len, D] del Perception Encoder de SAM 3
    en un vector compacto [B, 2*D] combinando Mean Pooling (contexto global)
    y Max Pooling (fenómenos locales salientes), ambos normalizados L2.
    """
    h = last_hidden_state.float()
    mean_pool = F.normalize(h.mean(dim=1), p=2, dim=-1)
    max_pool = F.normalize(h.amax(dim=1), p=2, dim=-1)
    return torch.cat([mean_pool, max_pool], dim=-1)


def extract_sam3_embeddings(frame: pd.DataFrame, batch_size: int = 4, desc: str = "Embeddings SAM 3") -> np.ndarray:
    """
    Extrae los embeddings del backbone visual de SAM 3 en batches.
    Omite el FPN neck y el decodificador de texto para máxima velocidad y bajo uso de VRAM.
    """
    filepaths = frame["filepath"].tolist()
    all_feats = []
    sam_model.eval()

    with torch.inference_mode():
        for start in tqdm(range(0, len(filepaths), batch_size), desc=desc):
            batch_paths = filepaths[start : start + batch_size]
            images = [Image.open(p).convert("RGB") for p in batch_paths]
            inputs = sam_processor(images=images, return_tensors="pt")
            pixel_values = inputs["pixel_values"].to(device=DEVICE, dtype=DTYPE)

            with torch.autocast(device_type="cuda", dtype=DTYPE, enabled=(DEVICE == "cuda")):
                backbone_out = sam_model.vision_encoder.backbone(pixel_values)

            pooled = pool_sam3_hidden_state(backbone_out.last_hidden_state)
            all_feats.append(pooled.cpu().numpy().astype(np.float32))

    return np.vstack(all_feats)


# Tamaño de batch seguro para RTX 3060 Ti (8 GB VRAM)
EXTRACT_BATCH_SIZE = 4
EMBEDS_CACHE_PATH = OUTPUTS_DIR / "sam3_embeddings.npz"

load_from_cache = False
if EMBEDS_CACHE_PATH.exists():
    cached = np.load(EMBEDS_CACHE_PATH)
    if (
        len(cached["X_train"]) == len(train_df)
        and len(cached["X_val"]) == len(val_df)
        and len(cached["X_test"]) == len(test_df)
    ):
        X_train = cached["X_train"]
        X_val = cached["X_val"]
        X_test = cached["X_test"]
        load_from_cache = True
        print(f"Embeddings de SAM 3 cargados instantáneamente desde caché: {EMBEDS_CACHE_PATH}")

if not load_from_cache:
    print("Extrayendo embeddings del Perception Encoder de SAM 3 (solo ocurre una vez y se guarda en caché)...")
    t0 = time.time()
    X_train = extract_sam3_embeddings(train_df, batch_size=EXTRACT_BATCH_SIZE, desc="SAM 3 Embeds [Train]")
    X_val = extract_sam3_embeddings(val_df, batch_size=EXTRACT_BATCH_SIZE, desc="SAM 3 Embeds [Val]")
    X_test = extract_sam3_embeddings(test_df, batch_size=EXTRACT_BATCH_SIZE, desc="SAM 3 Embeds [Test]")
    np.savez_compressed(EMBEDS_CACHE_PATH, X_train=X_train, X_val=X_val, X_test=X_test)
    elapsed = time.time() - t0
    print(f"Extracción completada en {elapsed / 60:.1f} min y guardada en {EMBEDS_CACHE_PATH}")

y_train = train_df["label_id"].values
y_val = val_df["label_id"].values
y_test = test_df["label_id"].values

print(f"Formas -> X_train: {X_train.shape} | X_val: {X_val.shape} | X_test: {X_test.shape}")

## 5. Bloque 3 — Entrenamiento de la Cabeza Clasificadora sobre SAM 3 (`Sam3WeatherHead`)

Con los *embeddings* visuales de SAM 3 ya calculados, entrenamos una cabeza MLP ligera (2 capas lineales con `LayerNorm`, `GELU` y `Dropout`) directamente sobre esas características.
- **0 CNNs externas:** toda la visión proviene 100% de SAM 3.
- **Entrenamiento ultrarrápido:** 40 épocas toman apenas **unos segundos** en GPU.
- Guardamos los pesos entrenados en `models/sam3_weather_head.pt`.

In [ ]:
class Sam3WeatherHead(nn.Module):
    """Cabeza clasificadora MLP ligera sobre los embeddings del Perception Encoder de SAM 3."""
    def __init__(self, in_dim: int, num_classes: int, hidden_dim: int = 512, dropout: float = 0.30):
        super().__init__()
        self.net = nn.Sequential(
            nn.LayerNorm(in_dim),
            nn.Dropout(dropout),
            nn.Linear(in_dim, hidden_dim),
            nn.GELU(),
            nn.LayerNorm(hidden_dim),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


EMBED_DIM = X_train.shape[1]
head_model = Sam3WeatherHead(in_dim=EMBED_DIM, num_classes=len(CLASSES), hidden_dim=512, dropout=0.30).to(DEVICE)

train_ds = TensorDataset(torch.from_numpy(X_train), torch.from_numpy(y_train).long())
val_ds = TensorDataset(torch.from_numpy(X_val), torch.from_numpy(y_val).long())
test_ds = TensorDataset(torch.from_numpy(X_test), torch.from_numpy(y_test).long())

train_loader = DataLoader(train_ds, batch_size=128, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=256, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=256, shuffle=False)

EPOCHS = 40
criterion = nn.CrossEntropyLoss(label_smoothing=0.05)
optimizer = torch.optim.AdamW(head_model.parameters(), lr=1e-3, weight_decay=1e-2)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

best_val_acc = 0.0
head_ckpt_path = MODELS_DIR / "sam3_weather_head.pt"
history = {"train_loss": [], "val_loss": [], "val_acc": []}

for epoch in range(1, EPOCHS + 1):
    # Entrenamiento
    head_model.train()
    running_loss = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        logits = head_model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * xb.size(0)
    scheduler.step()

    # Validación
    head_model.eval()
    val_loss = 0.0
    val_preds = []
    with torch.inference_mode():
        for xb, yb in val_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            logits = head_model(xb)
            val_loss += criterion(logits, yb).item() * xb.size(0)
            val_preds.extend(logits.argmax(dim=1).cpu().numpy())

    tr_loss = running_loss / len(train_df)
    va_loss = val_loss / len(val_df)
    va_acc = accuracy_score(y_val, val_preds)
    history["train_loss"].append(tr_loss)
    history["val_loss"].append(va_loss)
    history["val_acc"].append(va_acc)

    if va_acc > best_val_acc:
        best_val_acc = va_acc
        torch.save(
            {"model_state": head_model.state_dict(), "in_dim": EMBED_DIM, "classes": CLASSES},
            head_ckpt_path,
        )

    if epoch % 5 == 0 or epoch == 1:
        print(f"Época {epoch:02d}/{EPOCHS} | Train Loss: {tr_loss:.4f} | Val Loss: {va_loss:.4f} | Val Acc: {va_acc:.2%}")

print(f"\nMejor Val Accuracy (SAM 3 + Cabeza MLP): {best_val_acc:.2%} (guardado en {head_ckpt_path})")

# Cargar los mejores pesos
ckpt = torch.load(head_ckpt_path, map_location=DEVICE, weights_only=True)
head_model.load_state_dict(ckpt["model_state"])
head_model.eval()

# Curvas de entrenamiento
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
axes[0].plot(history["train_loss"], label="Train Loss")
axes[0].plot(history["val_loss"], label="Val Loss")
axes[0].set_title("Evolución de la pérdida (SAM 3 Head)")
axes[0].set_xlabel("Época")
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot([a * 100 for a in history["val_acc"]], color="#7c3aed")
axes[1].set_title("Exactitud en Validación (%)")
axes[1].set_xlabel("Época")
axes[1].set_ylabel("Accuracy (%)")
axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# 1) Evaluación en todo el set de Test (1,030 imágenes)
head_model.eval()
with torch.inference_mode():
    test_logits = head_model(torch.from_numpy(X_test).to(DEVICE))
    test_probs = torch.softmax(test_logits, dim=1).cpu().numpy()
    test_preds = test_probs.argmax(axis=1)

head_full_acc = accuracy_score(y_test, test_preds)
head_full_f1 = f1_score(y_test, test_preds, average="macro")

print(f"=== SAM 3 + Cabeza Clasificadora en Test Completo ({len(test_df):,} imágenes) ===")
print(f"Accuracy: {head_full_acc:.2%} | Macro F1: {head_full_f1:.2%}\n")
print(classification_report(y_test, test_preds, target_names=CLASSES, zero_division=0))

# 2) Comparativa directa en las mismas 110 imágenes de sam_zs_test_df
filepath_to_test_idx = {fp: i for i, fp in enumerate(test_df["filepath"].tolist())}
sub_indices = [filepath_to_test_idx[fp] for fp in sam_zs_test_df["filepath"].tolist()]
head_sub_preds = test_preds[sub_indices]
head_sub_acc = accuracy_score(y_zs, head_sub_preds)
head_sub_f1 = f1_score(y_zs, head_sub_preds, average="macro")

summary_df = pd.DataFrame([
    {
        "Configuración SAM 3": "SAM 3 Zero-Shot (sin entrenamiento)",
        "Muestra Test (110 imgs)": f"{zs_acc:.2%} (F1: {zs_f1:.2%})",
        "Test Completo (1,030 imgs)": "-",
        "Explicabilidad Visual": "Sí (máscaras SAM 3)",
    },
    {
        "Configuración SAM 3": "SAM 3 + Cabeza Clasificadora MLP (100% SAM 3)",
        "Muestra Test (110 imgs)": f"{head_sub_acc:.2%} (F1: {head_sub_f1:.2%})",
        "Test Completo (1,030 imgs)": f"{head_full_acc:.2%} (F1: {head_full_f1:.2%})",
        "Explicabilidad Visual": "Sí (mismo vision_embeds)",
    },
])
display(summary_df)

# Matrices de confusión lado a lado
fig, axes = plt.subplots(1, 2, figsize=(15.5, 6.2))
sns.heatmap(
    confusion_matrix(y_zs, pred_zs),
    annot=True, fmt="d", cmap="Blues", xticklabels=CLASSES, yticklabels=CLASSES, ax=axes[0]
)
axes[0].set_title(f"SAM 3 Zero-Shot — 110 imgs (Acc: {zs_acc:.1%})")
axes[0].set_xlabel("Predicción")
axes[0].set_ylabel("Real")

sns.heatmap(
    confusion_matrix(y_test, test_preds),
    annot=True, fmt="d", cmap="Purples", xticklabels=CLASSES, yticklabels=CLASSES, ax=axes[1]
)
axes[1].set_title(f"SAM 3 + Cabeza MLP — 1,030 imgs (Acc: {head_full_acc:.1%})")
axes[1].set_xlabel("Predicción")
axes[1].set_ylabel("Real")

plt.tight_layout()
plt.show()

## 6. Bloque 4 — Inferencia unificada y explicable (`diagnosticar_clima`)

Ahora unimos ambas capacidades de **SAM 3 en una sola pasada** sobre cualquier fotografía nueva:
1. `sam_model.get_vision_features(pixel_values)` calcula `vision_embeds` **una única vez**.
2. `Sam3WeatherHead` usa `vision_embeds.last_hidden_state` (Mean + Max Pooling L2) para predecir el clima con alta precisión.
3. El decodificador de SAM 3 **reutiliza esos mismos `vision_embeds`** únicamente con el *prompt* del clima predicho para dibujar la máscara explicativa en milisegundos.

In [ ]:
def diagnosticar_clima(image_path: str, true_label: str | None = None, score_threshold: float = 0.25):
    """
    Pipeline unificado 100% SAM 3:
    1) Codifica la imagen una sola vez (get_vision_features).
    2) Clasifica el clima con Sam3WeatherHead sobre vision_embeds.last_hidden_state.
    3) Reutiliza vision_embeds en el decodificador de SAM 3 para segmentar la evidencia visual.
    """
    img = Image.open(image_path).convert("RGB")
    img_inputs = sam_processor(images=img, return_tensors="pt")
    pixel_values = img_inputs["pixel_values"].to(device=DEVICE, dtype=DTYPE)
    original_sizes = img_inputs["original_sizes"].tolist()

    with torch.inference_mode(), torch.autocast(device_type="cuda", dtype=DTYPE, enabled=(DEVICE == "cuda")):
        # 1) Única pasada por el Perception Encoder + FPN Neck de SAM 3
        vision_embeds = sam_model.get_vision_features(pixel_values=pixel_values)

        # 2) Clasificación con la cabeza entrenada sobre SAM 3
        pooled_feat = pool_sam3_hidden_state(vision_embeds.last_hidden_state)
        logits = head_model(pooled_feat.float())
        probs = torch.softmax(logits, dim=1).cpu().numpy()[0]

    pred_idx = int(np.argmax(probs))
    pred_cls = CLASSES[pred_idx]
    pred_conf = float(probs[pred_idx])

    # 3) Explicabilidad visual: segmentar el clima ganador reutilizando vision_embeds
    prompt_text = WEATHER_PROMPTS[pred_cls]
    seg = segment_prompt_from_embeds(vision_embeds, original_sizes, prompt_text, score_threshold=score_threshold)
    # Si el umbral estándar no encontró píxeles, relajamos el umbral para mostrar la evidencia más probable
    if seg["area_ratio"] == 0.0 and score_threshold > 0.12:
        seg = segment_prompt_from_embeds(vision_embeds, original_sizes, prompt_text, score_threshold=0.12)

    area_pct = seg["area_ratio"] * 100
    presence_pct = seg["presence_prob"] * 100

    print(
        f"Diagnóstico SAM 3: {WEATHER_NAMES_ES[pred_cls]} ({pred_cls}) — Confianza: {pred_conf:.1%}\n"
        f"Evidencia visual (prompt='{prompt_text}'): presencia={presence_pct:.1f}% | área segmentada={area_pct:.1f}%"
        + (f" | Etiqueta real: {true_label}" if true_label else "")
    )

    # Visualización 3 paneles
    fig, axes = plt.subplots(1, 3, figsize=(15.5, 4.4))
    axes[0].imshow(img)
    axes[0].set_title("Imagen original" + (f" (Real: {true_label})" if true_label else ""))
    axes[0].axis("off")

    axes[1].imshow(overlay_mask(img, seg["mask"], color=(255, 80, 80), alpha=0.45))
    axes[1].set_title(f"Evidencia SAM 3: {WEATHER_NAMES_ES[pred_cls]} ({area_pct:.1f}%)")
    axes[1].axis("off")

    top3_idx = np.argsort(probs)[-3:][::-1]
    labels_top3 = [WEATHER_NAMES_ES[CLASSES[i]] for i in top3_idx][::-1]
    vals_top3 = [probs[i] for i in top3_idx][::-1]
    colors = ["#a78bfa", "#8b5cf6", "#6d28d9"][-len(vals_top3):]

    axes[2].barh(labels_top3, vals_top3, color=colors)
    axes[2].set_xlim(0, 1.05)
    axes[2].set_title("Top-3 probabilidades (SAM 3 + MLP)")
    axes[2].set_xlabel("Probabilidad")
    axes[2].grid(axis="x", linestyle="--", alpha=0.35)
    for i, v in enumerate(vals_top3):
        axes[2].text(min(v + 0.02, 0.90), i, f"{v:.1%}", va="center", fontsize=9)

    plt.tight_layout()
    plt.show()

# Probar con 4 imágenes aleatorias del conjunto de test
for _, row in test_df.sample(4, random_state=SEED).iterrows():
    diagnosticar_clima(row["filepath"], true_label=row["label"])

In [ ]:
# Prueba con tu propia foto local (descomenta y cambia la ruta)
# diagnosticar_clima("C:/ruta/a/tu/foto_del_clima.jpg")